# ДЗ 14. NumPy: векторизация, broadcasting, оси, маски 🔢
Решите задачи в Google Colab и сдайте через GitHub: папка `hw14` в репозитории `cyberformula3-homework`.

- **[База]** — обязательно для всех ·
- **[Челлендж]** — сколько получится ·
- **[Про]** — для тех, кому легко ·

Каждый ответ — **число + одна фраза**: что это значит для магазина.
Циклы `for` по заказам для расчётов не используем — только массивы и маски. Цикл можно там, где это сказано.

**Проверь себя перед сдачей:**
□ **Среда выполнения → Перезапустить и выполнить все** — ноутбук проходит без ошибок
□ Ответы совпадают с «Примером проверки», где он есть

In [ ]:
import numpy as np
# проверка: вставьте сюда строки 1–5 и запустите

## Загрузка данных — запустите как есть

In [2]:
# Загрузка: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
FILE = "orders.csv"
print("Работаем с файлом:", FILE)

Работаем с файлом: orders.csv


In [3]:
import csv
import json
import numpy as np

with open(FILE, encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

price, qty, city, category, status, month, product = [], [], [], [], [], [], []
for r in rows:
    price.append(int(r["price"]))
    qty.append(int(r["qty"]))
    city.append(r["city"])
    category.append(r["category"])
    status.append(r["status"])
    month.append(int(r["date"].split("-")[1]))
    product.append(r["product"])

price = np.array(price)
qty = np.array(qty)
city = np.array(city)
category = np.array(category)
status = np.array(status)
month = np.array(month)
product = np.array(product)

revenue = price * qty
paid = status == "оплачен"
print(revenue.shape, revenue[:5])

(500,) [198000  24000 159000 189000 129000]


## [База]

### Задача 1. Цены выросли на 5%
Магазин поднимает все цены на 5%. Посчитайте новые суммы заказов **одной строкой**, без цикла.
На сколько тенге вырастет общая выручка?

**Пример проверки:** выручка вырастет на `4 625 325 ₸`.

In [4]:
new_revenue = revenue * 1.05
diff = new_revenue.sum() - revenue.sum()
print(f"Выручка вырастет на {diff:,.0f} ₸".replace(",", " "))

Выручка вырастет на 4 625 325 ₸


### Задача 2. Смартфоны
С помощью маски по `category` найдите:
- сколько заказов в категории «Смартфоны»;
- на какую сумму;
- какая это доля от всех заказов.

**Пример проверки:** `131` заказ на `30 689 000 ₸`, доля `26.2%`.

In [5]:
mask_phones = category == "Смартфоны"
count_phones = mask_phones.sum()
sum_phones = revenue[mask_phones].sum()
share_phones = sum_phones / revenue.sum() * 100

print(f"{count_phones} заказ на {sum_phones:,.0f} ₸, доля {share_phones:.1f}% — категория смартфонов формирует заметную часть оборота магазина.".replace(",", " "))

131 заказ на 30 689 000 ₸  доля 33.2% — категория смартфонов формирует заметную часть оборота магазина.


### Задача 3. Два условия
Сколько **оплаченных** заказов из **Шымкента** и на какую сумму? Не забудьте скобки вокруг каждого условия.

**Пример проверки:** `71` заказ на `12 852 500 ₸`.

In [6]:
mask_shymkent_paid = (status == "оплачен") & (city == "Шымкент")
count_sp = mask_shymkent_paid.sum()
sum_sp = revenue[mask_shymkent_paid].sum()

print(f"{count_sp} заказ на {sum_sp:,.0f} ₸ — оплаченные заказы из Шымкента приносят стабильный подтверждённый доход.".replace(",", " "))

71 заказ на 12 852 500 ₸ — оплаченные заказы из Шымкента приносят стабильный подтверждённый доход.


### Задача 4. Неделя курьера
Первые 30 заказов — это 6 дней по 5 заказов: `w = revenue[:30].reshape(6, 5)`.
1. Сумма за каждый день — какую `axis` взять? Сколько чисел должно получиться?
2. В какой день (номер с 1) сумма самая большая?
3. Самый дорогой заказ **каждого** дня.

**Пример проверки:** лучший — **3-й** день, `1 094 000 ₸`; самые дорогие по дням — `[198000 319000 549000 449000 449000 319000]`.

In [8]:
w = revenue[:30].reshape(6, 5)

daily_sum = w.sum(axis=1)
best_day = daily_sum.argmax() + 1
best_sum = daily_sum.max()
daily_max = w.max(axis=1)

print(
    f"лучший — {best_day}-й день, {best_sum:,.0f} ₸; самые дорогие по дням — {daily_max} — "
    f"нагрузка на курьера распределена по дням неравномерно, с пиком в {best_day}-й день.".replace(",", " ")
)

лучший — 3-й день  1 094 000 ₸; самые дорогие по дням — [198000 319000 549000 449000 449000 319000] — нагрузка на курьера распределена по дням неравномерно  с пиком в 3-й день.


## [Челлендж]

### Задача 5. Заказы в трёх валютах
Покупатели из-за рубежа просят показать суммы заказов в своей валюте.
Курсы Нацбанка РК на 19 ноября 2026: 1 USD = 444.16 ₸, 1 EUR = 487.29 ₸, 1 RUB = 5.40 ₸.

1. Возьмите суммы **первых 5 заказов** из `revenue` и получите **одной операцией** таблицу «заказ × валюта» формы `(5, 3)`, округлите до 2 знаков.
   Какую форму нужно дать суммам заказов, чтобы сработал broadcasting?
2. Выведите рядом товар каждого из этих 5 заказов — массив `product` уже собран в ячейке загрузки.


**Пример проверки:** форма суммам — `(5, 1)`; первый заказ (Xiaomi Redmi Note 13, 198 000 ₸) — `[445.79  406.33  36666.67]`.

In [9]:
rates = np.array([444.16, 487.29, 5.40])
rev_5 = revenue[:5].reshape(5, 1)  # форма (5, 1) для broadcasting с (3,)
in_currencies = np.round(rev_5 / rates, 2)  # форма (5, 3), округление до 2 знаков

for prod, row_curr in zip(product[:5], in_currencies):
    print(f"{prod}: {row_curr}")

print(
    f"форма суммам — {rev_5.shape}; первый заказ ({product[0]}, {revenue[0]:,.0f} ₸) — {in_currencies[0]} — "
    f"пересчёт в валюты помогает оценивать спрос зарубежных покупателей в привычных им единицах.".replace(",", " ")
)

Xiaomi Redmi Note 13: [  445.79   406.33 36666.67]
Клавиатура Logitech K120: [  54.03   49.25 4444.44]
Samsung Galaxy Tab A9: [  357.98   326.29 29444.44]
Samsung Galaxy A55: [  425.52   387.86 35000.  ]
Монитор LG 27 дюймов: [  290.44   264.73 23888.89]
форма суммам — (5  1); первый заказ (Xiaomi Redmi Note 13  198 000 ₸) — [  445.79   406.33 36666.67] — пересчёт в валюты помогает оценивать спрос зарубежных покупателей в привычных им единицах.


### Задача 6. Категории
Для **каждой** категории из `np.unique(category)` выведите число заказов, выручку и средний чек (цикл по категориям можно).
Какая категория приносит больше всего денег? А у какой самый большой средний чек?

**Пример проверки:** и по выручке, и по среднему чеку лидируют **Ноутбуки** — `40 992 000 ₸`, средний чек ≈ `422 598 ₸`.

In [10]:
cats = np.unique(category)
best_rev_cat, best_rev = None, -1
best_avg_cat, best_avg = None, -1

for c in cats:
    m = category == c
    cnt = m.sum()
    rev = revenue[m].sum()
    avg = revenue[m].mean()
    print(f"{c}: заказов — {cnt}, выручка — {rev:,.0f} ₸, средний чек — {avg:,.0f} ₸".replace(",", " "))
    if rev > best_rev:
        best_rev, best_rev_cat = rev, c
    if avg > best_avg:
        best_avg, best_avg_cat = avg, c

print(
    f"по выручке лидирует {best_rev_cat} ({best_rev:,.0f} ₸), по среднему чеку — {best_avg_cat} ({best_avg:,.0f} ₸) — "
    f"крупная техника/ноутбуки задают основной объём оборота и высокий средний чек.".replace(",", " ")
)

Аксессуары: заказов — 171  выручка — 4 514 500 ₸  средний чек — 26 401 ₸
Мониторы: заказов — 50  выручка — 6 046 000 ₸  средний чек — 120 920 ₸
Ноутбуки: заказов — 97  выручка — 40 992 000 ₸  средний чек — 422 598 ₸
Планшеты: заказов — 51  выручка — 10 265 000 ₸  средний чек — 201 275 ₸
Смартфоны: заказов — 131  выручка — 30 689 000 ₸  средний чек — 234 267 ₸
по выручке лидирует Ноутбуки (40 992 000 ₸)  по среднему чеку — Ноутбуки (422 598 ₸) — крупная техника/ноутбуки задают основной объём оборота и высокий средний чек.


### Задача 7. Бесплатная доставка
Доставка бесплатна, если сумма заказа от 50 000 ₸. С помощью `np.where` подпишите каждый заказ: `"бесплатно"` или `"платно"`.
Сколько заказов получили бесплатную доставку?

**Пример проверки:** `354`.

In [ ]:
delivery = np.where(revenue >= 50000, "бесплатно", "платно")
free_count = (delivery == "бесплатно").sum()

print(
    f"{free_count} — большинство заказов проходит порог бесплатной доставки от 50 000 ₸, стимулируя более крупный чек.".replace(",", " ")
)

## [Про]

### Задача 8. Таблица «категория × месяц»
Постройте таблицу **числа заказов** формы `(5, 8)`: строки — категории (`np.unique(category)`), столбцы — месяцы (`np.unique(month)`).
Двойной цикл по категориям и месяцам можно, внутри — маска из двух условий.

1. Проверьте: сумма всей таблицы должна быть равна числу заказов.
2. В каком месяце было больше всего заказов смартфонов?

**Пример проверки:** сумма таблицы `500`; больше всего смартфонов — в **сентябре (9)**, `22` заказа.

In [11]:
cats = np.unique(category)
months = np.unique(month)
table = np.zeros((len(cats), len(months)), dtype=int)

for i, c in enumerate(cats):
    for j, m_val in enumerate(months):
        table[i, j] = ((category == c) & (month == m_val)).sum()

print("Таблица (категория x месяц), форма:", table.shape)
print(table)
print("Сумма всей таблицы:", table.sum(), "| Всего заказов:", len(revenue))

phone_idx = np.where(cats == "Смартфоны")[0][0]
best_m_idx = table[phone_idx].argmax()
best_month = months[best_m_idx]
best_phone_cnt = table[phone_idx, best_m_idx]

print(
    f"сумма таблицы {table.sum()}; больше всего смартфонов — в месяце {best_month}, {best_phone_cnt} заказа — "
    f"спрос на смартфоны имеет выраженную сезонность и достигает пика в {best_month}-м месяце."
)

Таблица (категория x месяц), форма: (5, 8)
[[17 13 22 13 22 38 24 22]
 [ 5  2  2  6  7 14  6  8]
 [13 12  8 14 11 13 19  7]
 [ 5  6  3  6  5 10 10  6]
 [14 13 16 15 19 19 22 13]]
Сумма всей таблицы: 500 | Всего заказов: 500
сумма таблицы 500; больше всего смартфонов — в месяце 9, 22 заказа — спрос на смартфоны имеет выраженную сезонность и достигает пика в 9-м месяце.


### Задача 9. Доли категорий по месяцам
Разделите таблицу на итог по **каждому месяцу** — получится доля каждой категории внутри месяца.
Это broadcasting: таблица `(5, 8)` делится на строку `(8,)`. Какая `axis` даёт итог по месяцам?

1. Проверьте: сумма каждого столбца после деления равна `1`.
2. Какая доля заказов в **августе** пришлась на аксессуары?
3. Сохраните доли в `shares.json`: `{"категории": [...], "месяцы": [...], "доли": [[...], ...]}`, доли округлите до 2 знаков. Не забудьте `.tolist()`.

**Пример проверки:** в августе аксессуары — `40.4%` заказов.

In [12]:
month_totals = table.sum(axis=0)        # форма (8,) — итог по каждому месяцу
shares = table / month_totals           # форма (5, 8) — broadcasting по столбцам

# 1. Проверка: сумма каждого столбца равна 1
print("Суммы по столбцам:", np.round(shares.sum(axis=0), 6))

# 2. Доля заказов в августе (месяц 8) на категорию "Аксессуары"
acc_idx = int(np.where(cats == "Аксессуары")[0][0])
aug_idx = int(np.where(months == 8)[0][0])
aug_acc_share_pct = shares[acc_idx, aug_idx] * 100

# 3. Сохранение в shares.json (модуль json уже импортирован в ячейке загрузки)
payload = {
    "категории": cats.tolist(),
    "месяцы": months.tolist(),
    "доли": np.round(shares, 2).tolist(),
}
with open("shares.json", "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print(
    f"в августе аксессуары — {aug_acc_share_pct:.1f}% заказов — "
    f"в августе аксессуары занимают заметную долю спроса и дополняют покупки основных товаров."
)

Суммы по столбцам: [1. 1. 1. 1. 1. 1. 1. 1.]
в августе аксессуары — 40.4% заказов — в августе аксессуары занимают заметную долю спроса и дополняют покупки основных товаров.
